# DL_bathy — inference-only demo

This notebook is a self-contained, Colab-ready walkthrough of running the
pretrained DL_bathy DeepLabV3+ model on Sentinel-2 imagery. It **only runs
inference** — no LiDAR ground truth is needed, no training happens, no
credentials are required.

The AOI is a small box in the Great Barrier Reef that matches the public
ground-truth subset shipped in the repo (`Ground_Truth.zip`), so the last
section compares predicted vs. reference bathymetry on overlapping pixels.

Pipeline in three lines of code:

1. Fetch a Sentinel-2 L2A scene over the AOI with `geoai-datacubes`.
2. Load the pretrained DeepLabV3+ weights from the Hugging Face Hub.
3. Predict per-pixel depth on the scene, plot, and compare to ground truth.


## Setup

In [ ]:
# One-time install in a fresh Colab kernel.
# Comment out the pip line if the packages are already in your local env.
%pip install --quiet DL_bathy git+https://github.com/buckai-observatory/DL_bathy.git \
                     geoai-datacubes huggingface_hub segmentation-models-pytorch \
                     rasterio matplotlib


In [ ]:
import os, sys, urllib.request, zipfile, io
import numpy as np
import rasterio
import matplotlib.pyplot as plt

# When running from a local clone, put the repo root on the path so the
# top-level `inference.py` is importable. In Colab, the pip install above
# already puts it on sys.path.
if "inference" not in sys.modules:
    sys.path.insert(0, os.getcwd())
import inference
print(f"inference module: {inference.__file__}")


## 1. Fetch a Sentinel-2 scene over the AOI

The demo AOI covers the same GBR reef area as the public ground-truth
subset in `Ground_Truth.zip`. Cloud filter is generous (< 20%) — reef
scenes routinely have some cloud cover, and the SCL mask will further
tighten the comparison later.


In [ ]:
AOI                = [145.14, -14.56, 145.66, -14.26]     # [lon_min, lat_min, lon_max, lat_max]
TIME_RANGE         = ("2020-06-01", "2020-09-30")
MAX_CLOUD_COVERAGE = 0.20


## 2. Load pretrained weights from the Hugging Face Hub

`inference.load_model()` downloads the checkpoint under the hood and
reconstructs the DeepLabV3+ architecture in one call. On the first run it
caches to `~/.cache/huggingface/` (a few hundred MB); subsequent calls
are near-instant.


In [ ]:
model = inference.load_model()
print(f"Loaded model on device: {next(model.parameters()).device}")


## 3. Fetch + predict in one call

The `predict_from_geoai_datacubes` helper wraps the whole
"fetch scene → run model → return depth map + provenance" flow.
For finer control (e.g. running many AOIs), call
`inference.predict_depth(model, s2_bands)` directly on a 12-band array
you already have on disk.


In [ ]:
depths, meta = inference.predict_from_geoai_datacubes(
    aoi                = AOI,
    time_range         = TIME_RANGE,
    model              = model,
    max_cloud_coverage = MAX_CLOUD_COVERAGE,
    save_folder        = "s2_fetched",
)
print(f"Predicted depth map: shape={depths.shape}, dtype={depths.dtype}")
print(f"Scene: {meta['scene_id']}")
print(f"Range: {np.nanmin(depths):.2f} m to {np.nanmax(depths):.2f} m")


## 4. Visualise the predicted depth map


In [ ]:
fig, ax = plt.subplots(figsize=(9, 7))
im = ax.imshow(depths, cmap="viridis_r", vmin=0, vmax=25)
ax.set_title(f"DL_bathy predicted depth [m]\n{meta['scene_id']}")
ax.set_xlabel("column"); ax.set_ylabel("row")
plt.colorbar(im, ax=ax, label="depth (m)")
plt.tight_layout(); plt.show()


## 5. Compare against public ground-truth bathymetry

`Ground_Truth.zip` in the repo root contains a bathymetric reference
raster for the same AOI. This section extracts it, reprojects to match
the predicted depth grid, and computes RMSE / MAE on the overlapping
valid pixels.


In [ ]:
GT_ZIP = "Ground_Truth.zip"

# In Colab, the zip lives at the repo root once we pip-installed from the
# GitHub source. If it is not present (e.g. the user is running from a
# vanilla env without cloning), pull it straight from the raw GitHub URL.
if not os.path.exists(GT_ZIP):
    url = "https://github.com/buckai-observatory/DL_bathy/raw/main/Ground_Truth.zip"
    print(f"Downloading {url} ...")
    urllib.request.urlretrieve(url, GT_ZIP)

with zipfile.ZipFile(GT_ZIP) as zf:
    print("Ground_Truth.zip contents:")
    for name in zf.namelist():
        print(f"  {name}")
    zf.extractall("Ground_Truth")

# Pick the first raster inside the unpacked directory. If Ground_Truth.zip
# ships more than one, adapt this to select the AOI-matching one.
import glob
gt_candidates = sorted(glob.glob("Ground_Truth/**/*.tif", recursive=True))
if not gt_candidates:
    raise FileNotFoundError("No .tif found inside Ground_Truth.zip")
gt_path = gt_candidates[0]
print(f"Using ground truth: {gt_path}")


In [ ]:
# Reproject ground truth onto the predicted-scene grid so we compare
# pixel-for-pixel. rasterio.warp handles the CRS / resolution mismatch.
from rasterio.warp import reproject, Resampling

with rasterio.open(gt_path) as src:
    gt_arr    = src.read(1).astype("float32")
    gt_crs    = src.crs
    gt_tf     = src.transform
    gt_nodata = src.nodata

gt_aligned = np.full_like(depths, np.nan, dtype="float32")
reproject(
    source        = gt_arr,
    destination   = gt_aligned,
    src_transform = gt_tf,      src_crs   = gt_crs,
    dst_transform = meta["transform"],  dst_crs = meta["crs"],
    src_nodata    = gt_nodata,   dst_nodata = np.nan,
    resampling    = Resampling.bilinear,
)

valid = np.isfinite(gt_aligned) & np.isfinite(depths)
n_valid = int(valid.sum())
print(f"Overlapping valid pixels: {n_valid:,}")
if n_valid == 0:
    print("!! No overlap between predicted scene and ground truth. "
          "Check the AOI / TIME_RANGE and the ground-truth raster.")


In [ ]:
# Side-by-side visual + summary metrics on the overlapping pixels.
fig, axes = plt.subplots(1, 3, figsize=(18, 6))

vmin, vmax = 0, 25

im0 = axes[0].imshow(depths, cmap="viridis_r", vmin=vmin, vmax=vmax)
axes[0].set_title("DL_bathy prediction [m]"); axes[0].axis("off")
plt.colorbar(im0, ax=axes[0], fraction=0.046)

im1 = axes[1].imshow(gt_aligned, cmap="viridis_r", vmin=vmin, vmax=vmax)
axes[1].set_title("Ground truth (public subset) [m]"); axes[1].axis("off")
plt.colorbar(im1, ax=axes[1], fraction=0.046)

diff = depths - gt_aligned
im2 = axes[2].imshow(diff, cmap="RdBu_r", vmin=-5, vmax=5)
axes[2].set_title("Prediction − ground truth [m]"); axes[2].axis("off")
plt.colorbar(im2, ax=axes[2], fraction=0.046)

plt.tight_layout(); plt.show()

if valid.any():
    err  = depths[valid] - gt_aligned[valid]
    rmse = float(np.sqrt(np.mean(err ** 2)))
    mae  = float(np.mean(np.abs(err)))
    bias = float(np.mean(err))
    print(f"RMSE = {rmse:.2f} m   MAE = {mae:.2f} m   bias = {bias:+.2f} m   n = {int(valid.sum()):,}")


## Next steps

- Change `AOI` / `TIME_RANGE` at the top of the notebook to point at your
  own region. Any Sentinel-2 covered coastline works; the model was
  trained on GBR-style shallow reef, so accuracy will degrade in visually
  different environments (turbid coasts, high sun-glint, etc.).
- For batch inference over many AOIs, use the lower-level
  `inference.load_model(...)` + `inference.predict_depth(...)` pair.
- To retrain on your own ground truth: `step1_splitting.py` prepares the
  patches, `step2_train_test.py` runs the training and evaluation. See
  the README for the full workflow.
